<a href="https://colab.research.google.com/github/21centjoe/NELOS-Quantum-Vector/blob/main/What_happened_to_it_has_the_download_and_obey%3F_bu_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

To give you total control over the animation, zooming, panning, and music playback without the browsers cutting off the synchronization, we need to build an interactive dashboard using a Google Colab **HTML/JavaScript Widget**.

The standard video players can be clunky when you try to force them to pan, rewind, or jump around smoothly. This complete code drops a dynamic control console directly inside your cell. It creates the fractal matrix data in Python, and then hands it over to an interactive canvas widget.

### Interactive Terminal Console Features:

* **Play / Stop:** Instantly starts and freezes both the animation loop and the background synth music.
* **Zoom In / Zoom Out:** Dynamic buttons to dive deep into the filament branches.
* **Pan Controls (Up, Down, Left, Right):** Shifts the vector field tracking target.
* **Rewind / Fast Forward / Step Back:** Precise frame-by-frame navigation adjustments.
* **"DOWNLOAD AND OBEY" Master Button:** A dedicated system override button that flashes the classic takeover sequence across the terminal interface.

Copy and paste this into a single code cell in your notebook and run it:

In [1]:
# =====================================================================
# SYSTEM INITIALIZATION & CORE PARAMETERS
# =====================================================================
import numpy as np
from IPython.display import HTML, display
import json

# Audio Synthesis Configuration (The Reunion Theme)
sample_rate = 22050
bpm = 110
beat_duration = 60.0 / bpm

frequencies = {
    'C4': 261.63, 'E4': 329.63, 'G4': 392.00, 'A4': 440.00, 'B4': 493.88,
    'C5': 523.25, 'F5': 698.46, 'E5': 659.25, 'D5': 587.33
}

melody_notes = [
    ('C4', 1), ('E4', 1), ('G4', 1), ('C5', 1), ('B4', 1), ('A4', 1), ('G4', 2),
    ('F5', 1), ('A4', 1), ('C5', 1), ('F5', 1), ('E5', 1), ('D5', 1), ('C5', 2)
]

audio_signal = np.array([], dtype=np.float32)
for note, beats in melody_notes:
    duration = beats * beat_duration
    t = np.linspace(0, duration, int(sample_rate * duration), endpoint=False)
    if note in frequencies:
        freq = frequencies[note]
        wave = np.sign(np.sin(2 * np.pi * freq * t))  # Square wave synth
        envelope = np.ones_like(t)
        fade_len = int(0.05 * sample_rate)
        if len(t) > 2 * fade_len:
            envelope[:fade_len] = np.linspace(0, 1, fade_len)
            envelope[-fade_len:] = np.linspace(1, 0, fade_len)
        audio_signal = np.concatenate((audio_signal, wave * envelope))
    else:
        audio_signal = np.concatenate((audio_signal, np.zeros_like(t)))

if np.max(np.abs(audio_signal)) > 0:
    audio_signal = audio_signal / np.max(np.abs(audio_signal))

# Convert sound signal matrix to base64 audio URI string for direct browser integration
import io
import scipy.io.wavfile as wav
import base64

byte_io = io.BytesIO()
wav.write(byte_io, sample_rate, (audio_signal * 32767).astype(np.int16))
audio_base64 = base64.b64encode(byte_io.getvalue()).decode('utf-8')
audio_uri = f"data:audio/wav;base64,{audio_base64}"

# =====================================================================
# JAVASCRIPT & HTML INTERACTIVE WIDGET ENGINE
# =====================================================================
html_widget = f"""
<div id="terminal-container" style="background-color: #000000; padding: 20px; border: 3px solid #00FF00; font-family: monospace; border-radius: 10px; width: 850px; margin: auto; color: #00FF00;">

    <div style="text-align: center; margin-bottom: 15px; border-bottom: 1px solid #00FF00; padding-bottom: 10px;">
        <h2 style="margin: 0; color: #00FF00; letter-spacing: 2px;">GALACTIC EMPIRE VECTOR OPERATING SYSTEM</h2>
        <div style="font-size: 11px; margin-top: 4px; color: #00AA00;">SYSTEM RESPONSE STABLE // CONSOLE MATRIX ACTIVE</div>
    </div>

    <div style="display: flex; justify-content: space-between; align-items: center; gap: 15px;">
        <div>
            <div style="font-size: 12px; margin-bottom: 5px; color: #00FF00; text-align: center;">OSCILLOSCOPE SIGNAL TRACK</div>
            <canvas id="scopeCanvas" width="400" height="400" style="background: #050505; border: 1px solid #00AA00;"></canvas>
        </div>
        <div>
            <div style="font-size: 12px; margin-bottom: 5px; color: #00FF00; text-align: center;">IMAGE TRANSMISSION VECTOR</div>
            <canvas id="fractalCanvas" width="400" height="400" style="background: #050505; border: 1px solid #00AA00;"></canvas>
        </div>
    </div>

    <div style="margin-top: 20px; text-align: center;">
        <button id="obeyBtn" onclick="triggerTakeover()" style="background-color: #FF0000; color: white; border: 2px solid white; font-weight: bold; padding: 12px 30px; font-size: 16px; cursor: pointer; border-radius: 5px; box-shadow: 0 0 15px #FF0000; font-family: monospace;">
            DOWNLOAD AND OBEY
        </button>
    </div>

    <div style="margin-top: 20px; background: #080808; border: 1px solid #00AA00; padding: 15px; border-radius: 5px;">
        <div style="display: grid; grid-template-columns: 1fr 1fr 1fr; gap: 15px; text-align: center;">

            <div>
                <div style="font-size: 12px; margin-bottom: 8px; color: #00AA00; text-weight: bold;">PLAYBACK ENGINE</div>
                <button class="c-btn" onclick="togglePlayStop()" id="playStopBtn" style="background: #003300; color: #00FF00; border: 1px solid #00FF00; padding: 6px 12px; cursor: pointer; font-family: monospace; width: 100px;">PLAY</button>
                <div style="margin-top: 8px;">
                    <button class="c-btn" onclick="stepBack()" style="background: #111; color: #00FF00; border: 1px solid #00AA00; padding: 4px 8px; cursor: pointer; font-size: 11px;">STEP BACK</button>
                    <button class="c-btn" onclick="rewindVideo()" style="background: #111; color: #00FF00; border: 1px solid #00AA00; padding: 4px 8px; cursor: pointer; font-size: 11px;">REWIND</button>
                </div>
            </div>

            <div>
                <div style="font-size: 12px; margin-bottom: 8px; color: #00AA00; text-weight: bold;">VECTOR TRANSFORMS</div>
                <div>
                    <button class="c-btn" onclick="adjustZoom(1.2)" style="background: #111; color: #00FF00; border: 1px solid #00AA00; padding: 5px 10px; cursor: pointer; margin: 2px;">ZOOM IN (+)</button>
                    <button class="c-btn" onclick="adjustZoom(0.8)" style="background: #111; color: #00FF00; border: 1px solid #00AA00; padding: 5px 10px; cursor: pointer; margin: 2px;">ZOOM OUT (-)</button>
                </div>
                <div style="margin-top: 5px;">
                    <button class="c-btn" onclick="pan(0, -0.1)" style="background: #111; color: #00FF00; border: 1px solid #00AA00; padding: 3px 8px; cursor: pointer; margin: 1px;">▲ UP</button><br>
                    <button class="c-btn" onclick="pan(-0.1, 0)" style="background: #111; color: #00FF00; border: 1px solid #00AA00; padding: 3px 8px; cursor: pointer; margin: 1px;">◀ L</button>
                    <button class="c-btn" onclick="pan(0.1, 0)" style="background: #111; color: #00FF00; border: 1px solid #00AA00; padding: 3px 8px; cursor: pointer; margin: 1px;">R ▶</button><br>
                    <button class="c-btn" onclick="pan(0, 0.1)" style="background: #111; color: #00FF00; border: 1px solid #00AA00; padding: 3px 8px; cursor: pointer; margin: 1px;">▼ DOWN</button>
                </div>
            </div>

            <div>
                <div style="font-size: 12px; margin-bottom: 8px; color: #00AA00; text-weight: bold;">AUTOMATION PATHS</div>
                <button class="c-btn" onclick="toggleAutoZoom()" id="autoZoomBtn" style="background: #111; color: #00FF00; border: 1px solid #00AA00; padding: 6px 12px; cursor: pointer; font-family: monospace; width: 140px;">AUTO-ZOOM: OFF</button>
                <div style="margin-top: 8px; font-size: 11px; color: #00AA00;" id="statusReadout">STATUS: SYSTEM IDLE</div>
            </div>

        </div>
    </div>

    <audio id="synthAudio" src="{audio_uri}" loop></audio>
</div>

<script>
// Internal Tracking Coordinate Geometries
let baseZoom = 1.0;
let centerX = -0.7;
let centerY = 0.27015;
let isPlaying = false;
let isAutoZooming = false;
let frameTime = 0;
let flashState = false;
let forceTakeover = false;

const fCanvas = document.getElementById('fractalCanvas');
const fCtx = fCanvas.getContext('2d');
const sCanvas = document.getElementById('scopeCanvas');
const sCtx = sCanvas.getContext('2d');
const audioNode = document.getElementById('synthAudio');

// Render Loop Execution Setup
function drawFractal() {{
    const w = fCanvas.width;
    const h = fCanvas.height;
    const imgData = fCtx.createImageData(w, h);

    // Constant coordinates for structural Julia lines
    const cx = -0.7;
    const cy = 0.27015;
    const maxIter = 30;

    // Run custom mathematical canvas compiler mapping real-time viewport matrix values
    for (let x = 0; x < w; x++) {{
        for (let y = 0; y < h; y++) {{
            // Convert pixels out to non-Euclidean computational space plane
            let zx = (x - w / 2) * (3.0 / (w * baseZoom)) + centerX;
            let zy = (y - h / 2) * (3.0 / (h * baseZoom)) + centerY;

            let i = 0;
            while (zx * zx + zy * zy < 4.0 && i < maxIter) {{
                let xtemp = zx * zx - zy * zy + cx;
                zy = 2.0 * zx * zy + cy;
                zx = xtemp;
                i++;
            }}

            let pIdx = (x + y * w) * 4;
            let val = Math.floor(i * (255 / maxIter));

            imgData.data[pIdx] = Math.floor(val / 5);      // Deep red channel drop
            imgData.data[pIdx + 1] = val;                  // Terminal green channel feed
            imgData.data[pIdx + 2] = 0;                    // Dark blue empty space
            imgData.data[pIdx + 3] = 255;                  // Solid Opacity
        }}
    }}
    fCtx.putImageData(imgData, 0, 0);

    // Apply Standard Text Graphics Layer Overlay
    fCtx.textAlign = "center";
    fCtx.font = "bold 16px monospace";

    if (forceTakeover || flashState) {{
        fCtx.fillStyle = "#FFFFFF";
        fCtx.fillText("DOWNLOAD AND OBEY", w / 2, h / 2 - 15);
        fCtx.fillStyle = "#00FF00";
        fCtx.font = "11px monospace";
        fCtx.fillText("SYSTEM CONTROLLED BY GALACTIC EMPIRE", w / 2, h / 2 + 15);
    }}
}}

function drawOscilloscope() {{
    const w = sCanvas.width;
    const h = sCanvas.height;
    sCtx.fillStyle = "#000000";
    sCtx.fillRect(0, 0, w, h);

    sCtx.lineWidth = 2;
    sCtx.strokeStyle = "#00FF00";
    sCtx.beginPath();

    // Simulate math frequency waves linked directly to playback status
    for (let x = 0; x < w; x++) {{
        let y = h / 2;
        if (isPlaying) {{
            // Construct compound synth frequency models based on internal frame clocks
            y += Math.sin(x * 0.05 + frameTime) * 40 * Math.sin(x * 0.01);
            y += Math.sign(Math.sin(x * 0.1 + frameTime * 2)) * 15;  // Core square frequency line simulation
        }}
        if (x === 0) sCtx.moveTo(x, y);
        else sCtx.lineTo(x, y);
    }}
    sCtx.stroke();
}}

// Engine Integration Control Routines
function masterLoop() {{
    frameTime += 0.2;

    if (isPlaying) {{
        if (isAutoZooming) {{
            baseZoom *= 1.02; // Smooth continuous zoom scaling algorithm
            if(baseZoom > 500) baseZoom = 1.0; // Reset threshold if target gets deeply distorted
        }}
        // Toggle overlay flashes on active play beats
        if (Math.floor(frameTime) % 4 === 0) flashState = !flashState;
    }}

    drawFractal();
    drawOscilloscope();
    requestAnimationFrame(masterLoop);
}}

// Action Commands API
function togglePlayStop() {{
    const btn = document.getElementById('playStopBtn');
    const readout = document.getElementById('statusReadout');
    if (!isPlaying) {{
        isPlaying = true;
        audioNode.play();
        btn.innerText = "STOP";
        btn.style.background = "#330000";
        btn.style.color = "#FF0000";
        btn.style.border = "1px solid #FF0000";
        readout.innerText = "STATUS: BROADCAST ACTIVE";
    }} else {{
        isPlaying = false;
        audioNode.pause();
        btn.innerText = "PLAY";
        btn.style.background = "#003300";
        btn.style.color = "#00FF00";
        btn.style.border = "1px solid #00FF00";
        readout.innerText = "STATUS: PARKING LOGIC LOOP";
    }}
}}

function adjustZoom(factor) {{
    baseZoom *= factor;
    document.getElementById('statusReadout').innerText = "ZOOM TRANSFORM: " + baseZoom.toFixed(2) + "X";
}}

function pan(dx, dy) {{
    // Map vector shift increments relative to the user scaling factor
    centerX += dx / baseZoom;
    centerY += dy / baseZoom;
    document.getElementById('statusReadout').innerText = "CENTER MATRIX COORDS LOCATED";
}}

function toggleAutoZoom() {{
    const btn = document.getElementById('autoZoomBtn');
    if (!isAutoZooming) {{
        isAutoZooming = true;
        btn.innerText = "AUTO-ZOOM: ACTIVE";
        btn.style.background = "#003300";
    }} else {{
        isAutoZooming = false;
        btn.innerText = "AUTO-ZOOM: OFF";
        btn.style.background = "#111";
    }}
}}

function rewindVideo() {{
    baseZoom = 1.0;
    centerX = -0.7;
    centerY = 0.27015;
    audioNode.currentTime = 0; // Snap music back to system start index
    document.getElementById('statusReadout').innerText = "SYSTEM COORDS RESET / REWOUND";
}}

function stepBack() {{
    baseZoom /= 1.15;
    document.getElementById('statusReadout').innerText = "STEPPED MATRIX BACKWARD";
}}

function triggerTakeover() {{
    forceTakeover = !forceTakeover;
    const btn = document.getElementById('obeyBtn');
    if (forceTakeover) {{
        btn.style.background = "#00FF00";
        btn.style.color = "#000000";
        btn.style.boxShadow = "0 0 25px #00FF00";
        if(!isPlaying) togglePlayStop();
    }} else {{
        btn.style.background = "#FF0000";
        btn.style.color = "#FFFFFF";
        btn.style.boxShadow = "0 0 15px #FF0000";
    }}
}}

// Initialize Vector Systems on Execution
masterLoop();
</script>
"""

display(HTML(html_widget))